# PA2 stage 04: persona LoRA-DPO with TRL (Task 5)

`HuggingFaceTB/SmolLM2-360M-Instruct` + LoRA trained with TRL `DPOTrainer` on
`data/persona_lora/ramsay_style_preferences.json` (200 train / 20 held out). v1 is the first
configuration (over-optimised); v2-v4 lower the learning rate, add an SFT term on the chosen
response, and raise beta (too weak to change generations); v5-v6 restore update strength with the SFT anchor. Every variant and the untuned model answer the fixed 40-prompt
persona style set with identical greedy decoding.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --format="%h %s"

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "peft==0.21.2" "trl==1.15.0"
!python -m pip install -q -e . --no-deps

In [ ]:
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2"

In [ ]:
# Let PyTorch grow memory segments instead of fragmenting (inherited by every ! and subprocess call).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Link the attached prepared-data dataset to data/prepared, wherever Kaggle mounted it.
import glob
hits = glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)
assert hits, "attach the pa2-alignment-data dataset"
DATA_DIR = os.path.dirname(hits[0])
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/prepared"):
    os.symlink(DATA_DIR, "data/prepared")
!ls -la data/prepared/
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
# Resume support: copy this stage's run folders back from a previous run's attached output.
import shutil
for run in ['lora_dpo', 'lora_dpo_v2', 'lora_dpo_v3', 'lora_dpo_v4', 'lora_dpo_v5', 'lora_dpo_v6', 'gen_persona']:
    local = os.path.join("outputs", run)
    if os.path.exists(local):
        continue
    found = sorted(glob.glob(f"/kaggle/input/**/small-model-alignment-starter/outputs/{run}", recursive=True))
    if found:
        print(f"restoring outputs/{run} from {found[-1]}")
        shutil.copytree(found[-1], local, symlinks=False)
    else:
        print(f"outputs/{run}: starting fresh")
!du -sh outputs/* 2>/dev/null || true

In [ ]:
# Each variant trains (skipped if its summary.json exists), then answers the persona set (finished rows skipped).
# GPU 1 also generates the untuned SmolLM2 baseline. Two chains run in parallel.
VARIANTS = [['lora_dpo', 'configs/lora_dpo.yaml', 'smollm2_lora_dpo'], ['lora_dpo_v2', 'configs/lora_dpo_v2.yaml', 'smollm2_lora_dpo_v2'], ['lora_dpo_v3', 'configs/lora_dpo_v3.yaml', 'smollm2_lora_dpo_v3'], ['lora_dpo_v4', 'configs/lora_dpo_v4.yaml', 'smollm2_lora_dpo_v4'], ['lora_dpo_v5', 'configs/lora_dpo_v5.yaml', 'smollm2_lora_dpo_v5'], ['lora_dpo_v6', 'configs/lora_dpo_v6.yaml', 'smollm2_lora_dpo_v6']]
gen = ("--eval_set eval_sets/persona_style_eval_40.json --batch_size 20 --max_new_tokens 256 --chat_template")

def variant_chain(run, config, stem):
    return (f"mkdir -p outputs/{run} && python -u scripts/train_lora_dpo.py --config {config} >> outputs/{run}/train.log 2>&1 && "
            f"python -u scripts/generate.py --model outputs/{run}/merged --generator smollm2-360m-{run.replace('_', '-')} "
            f"--out outputs/gen_persona/{stem}.json {gen}")

base_gen = (f"python -u scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --generator smollm2-360m-instruct "
            f"--out outputs/gen_persona/smollm2_base.json {gen}")
chains = {"0": [variant_chain(*v) for v in VARIANTS[0::2]], "1": [base_gen] + [variant_chain(*v) for v in VARIANTS[1::2]]}
os.makedirs("outputs/gen_persona", exist_ok=True)
procs = {gpu: subprocess.Popen(" && ".join(cmds), shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONPATH": "."},
                               stdout=open(f"outputs/gen_persona/gpu{gpu}.log", "a"), stderr=subprocess.STDOUT)
         for gpu, cmds in chains.items()}
print("exit codes", {gpu: p.wait() for gpu, p in procs.items()})
!tail -n 4 outputs/gen_persona/gpu0.log outputs/gen_persona/gpu1.log
for run, _, _ in VARIANTS:
    print(f"== {run}"); os.system(f"grep -E 'trainable|Traceback|Error' outputs/{run}/train.log | tail -3")

In [ ]:
import json
for run, _, _ in VARIANTS:
    os.system(f"PYTHONPATH=. python scripts/plot_metrics.py outputs/{run}")
    s = json.load(open(f"outputs/{run}/summary.json"))
    f = s["final_eval"]
    print(f"{run:12s} steps={s['optimizer_steps']} eval_acc={f['eval_rewards/accuracies']:.3f} "
          f"margin={f['eval_rewards/margins']:.3f} loss={f['eval_loss']:.3f}")
outs = {stem: {r["id"]: r for r in json.load(open(f"outputs/gen_persona/{stem}.json"))} for stem in
        ["smollm2_base"] + [v[2] for v in VARIANTS]}
for stem, rows in outs.items():
    n_len = sum(r["finish_reason"] == "length" for r in rows.values())
    print(f"{stem:22s} hit_max_tokens={n_len:2d} mean_words={sum(len(r['output'].split()) for r in rows.values()) / len(rows):.0f}")
for rid in list(outs["smollm2_base"])[::8]:
    print("\n###", outs["smollm2_base"][rid]["category"], "|", outs["smollm2_base"][rid]["instruction"])
    for stem, rows in outs.items():
        print(f"  [{stem}]", rows[rid]["output"][:300].replace("\n", " "))

In [ ]:
!cd outputs && zip -qr /kaggle/working/lora_dpo_run_outputs.zip lora_dpo lora_dpo_v2 lora_dpo_v3 lora_dpo_v4 lora_dpo_v5 lora_dpo_v6 gen_persona -x "*/merged/*" "*/trainer/*"
!du -sh outputs/lora_dpo* /kaggle/working/lora_dpo_run_outputs.zip